# 🧠 DSA Lab 13: Trees and Binary Trees: Terminology, Representation and Traversals
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 13 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 🚀 Motivation: Data with a Hierarchy

**A real hierarchy is a tree**

```text
                   University of Okara
                  /         |          \
        Faculty of CS    Faculty of     Faculty of
         & IT            Sciences       Management
        /    |    \
    BSCS   BSIT   BSSE
   /    \
3rd-A   3rd-B
```

## 📖 Part A: Tree Terminology

**Example tree used in this lab**

```text
     A            <- root (level 0, depth 0)
   /   \
  B     C         <- level 1   (B and C are siblings)
 / \     \
D   E     F       <- level 2
   / \
  G   H           <- level 3   (G, H, D, F are leaves)
```

| Term | Meaning | In the example |
|---|---|---|
| Node | An element of the tree | A, B, … H (8 nodes) |
| Edge | A link between parent and child | 7 edges (always n − 1) |
| Root | The top node with no parent | A |
| Parent / Child | A node directly above / below another | B is parent of D and E |
| Siblings | Nodes with the same parent | D and E; G and H |
| Leaf | A node with no children | D, G, H, F |
| Internal node | A node with at least one child | A, B, C, E |
| Degree | Number of children of a node | deg(B) = 2, deg(C) = 1 |
| Depth of a node | Edges from the root to the node | depth(E) = 2 |
| Height of a node | Edges on the longest path down to a leaf | height(B) = 2 |
| Height of the tree | Height of the root | 3 |
| Subtree | A node and all its descendants | B, D, E, G, H |
| Ancestor / Descendant | Nodes on the path up / below | A, B, E are ancestors of G |

## 🌲 Part B: Binary Trees

A **binary tree** is a tree where every node has **at most two children**, called the **left child** and the **right child**.

| Type | Definition | Example use |
|---|---|---|
| Full (strict) | Every node has 0 or 2 children | Expression trees, Huffman trees |
| Complete | All levels full except possibly the last, which is filled left to right | Heaps (Lab 15) |
| Perfect | All internal nodes have 2 children and all leaves are at the same level | Tournament brackets |
| Balanced | Left and right subtree heights differ by at most 1 at every node | AVL / Red-Black trees |
| Degenerate (skewed) | Every node has only one child; behaves like a linked list | Worst case for BSTs |

### Important Properties

- Maximum nodes at level k = **2ᵏ**.
- Maximum nodes in a binary tree of height h = **2ʰ⁺¹ − 1** (a perfect tree).
- Minimum height of a binary tree with n nodes = **⌊log₂ n⌋**. This is why balanced trees give O(log n) operations.
- In a full binary tree, number of leaves = number of internal nodes + 1.

### Representation 1: Linked Nodes

In [ ]:
class TreeNode:
    def __init__(self, data, left=None, right=None):
        self.data = data
        self.left = left
        self.right = right

def build_tree(values):
    """Build a binary tree from a level-order list; None means 'no node'."""
    if not values or values[0] is None:
        return None
    from collections import deque
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

# Build the example tree by hand
G, H = TreeNode("G"), TreeNode("H")
D, E = TreeNode("D"), TreeNode("E", G, H)
F = TreeNode("F")
B_, C_ = TreeNode("B", D, E), TreeNode("C", None, F)
root = TreeNode("A", B_, C_)

def show(node, indent="", label="root"):
    """Print the tree sideways: right subtree on top."""
    if node is None:
        return
    show(node.right, indent + "        ", "R")
    print(f"{indent}{label}: {node.data}")
    show(node.left, indent + "        ", "L")

show(root)

### Representation 2: Array (for Complete Trees)

Store nodes level by level in an array. For the node at index **i** (0-based): left child at **2i + 1**, right child at **2i + 2**, parent at **(i − 1) // 2**. No pointers needed; this is exactly how heaps are stored in Lab 15.

In [ ]:
tree = ["A", "B", "C", "D", "E", "F", "G"]     # a complete tree
i = 1                                            # node "B"
print("node:", tree[i], "| left:", tree[2 * i + 1], "| right:", tree[2 * i + 2],
      "| parent:", tree[(i - 1) // 2])

## 🚶 Part C: Tree Traversals

A **traversal** visits every node exactly once. Depth-first traversals differ only in **when** the current node is visited relative to its subtrees:

| Traversal | Order | Result on the example tree | Typical use |
|---|---|---|---|
| Preorder | Node → Left → Right | A B D E G H C F | Copy a tree, print folder structure |
| Inorder | Left → Node → Right | D B G E H A C F | Sorted order in a BST (Lab 14) |
| Postorder | Left → Right → Node | D G H E B F C A | Delete a tree, compute folder sizes |
| Level order | Level by level, left to right | A B C D E F G H | Shortest paths, printing by levels |

> **🧠 Memory trick**
>
> The name tells you where the **node** goes: **Pre** = node first, **In** = node in the middle, **Post** = node last. Left always comes before right.

### Recursive Depth-First Traversals

In [ ]:
def preorder(node, out):
    if node:
        out.append(node.data)           # visit
        preorder(node.left, out)
        preorder(node.right, out)
    return out

def inorder(node, out):
    if node:
        inorder(node.left, out)
        out.append(node.data)           # visit
        inorder(node.right, out)
    return out

def postorder(node, out):
    if node:
        postorder(node.left, out)
        postorder(node.right, out)
        out.append(node.data)           # visit
    return out

print("pre :", " ".join(preorder(root, [])))
print("in  :", " ".join(inorder(root, [])))
print("post:", " ".join(postorder(root, [])))

### Level-Order Traversal with a Queue (Breadth-First)

In [ ]:
from collections import deque

def level_order(root):
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        out.append(node.data)
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return out

print("level:", " ".join(level_order(root)))

### Iterative Preorder with a Stack

Recursion uses the call stack; we can use our own stack instead. Push the right child **before** the left child so that the left one is processed first.

In [ ]:
def preorder_iter(root):
    if root is None:
        return []
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node.data)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out

print("pre (stack):", " ".join(preorder_iter(root)))

## 🔁 Part D: Recursive Thinking on Trees

Almost every tree problem follows one pattern: **solve for the left subtree, solve for the right subtree, combine with the current node**. The base case is the empty tree (`None`).

In [ ]:
def size(node):
    return 0 if node is None else 1 + size(node.left) + size(node.right)

def height(node):
    if node is None:
        return -1                       # empty tree
    return 1 + max(height(node.left), height(node.right))

def count_leaves(node):
    if node is None:
        return 0
    if node.left is None and node.right is None:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)

print("size:", size(root), "| height:", height(root), "| leaves:", count_leaves(root))

## 🧮 Part E: Expression Trees

An arithmetic expression can be stored as a binary tree: **leaves are numbers** and **internal nodes are operators**. Inorder traversal (with brackets) gives the infix form, postorder gives the postfix form (Lab 10), and evaluating is a postorder computation.

**Expression tree**

```text
     *
   /   \
  +     -          represents  (3 + 5) * (8 - 2)
 / \   / \
3   5 8   2
```

In [ ]:
expr = TreeNode("*", TreeNode("+", TreeNode(3), TreeNode(5)),
                     TreeNode("-", TreeNode(8), TreeNode(2)))

def evaluate(node):
    if node.left is None and node.right is None:
        return node.data                         # a number
    a, b = evaluate(node.left), evaluate(node.right)
    return {"+": a + b, "-": a - b, "*": a * b, "/": a / b}[node.data]

def to_infix(node):
    if node.left is None and node.right is None:
        return str(node.data)
    return f"({to_infix(node.left)} {node.data} {to_infix(node.right)})"

print(to_infix(expr), "=", evaluate(expr))
print("postfix:", " ".join(str(x) for x in postorder(expr, [])))

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <queue>
#include <algorithm>
using namespace std;

struct Node {
    char data; Node* left; Node* right;
    Node(char d, Node* l = nullptr, Node* r = nullptr) : data(d), left(l), right(r) {}
};

void inorder(Node* n) { if (!n) return; inorder(n->left); cout << n->data << " "; inorder(n->right); }
int height(Node* n) { return n ? 1 + max(height(n->left), height(n->right)) : -1; }

void levelOrder(Node* root) {
    queue<Node*> q; if (root) q.push(root);
    while (!q.empty()) {
        Node* n = q.front(); q.pop();
        cout << n->data << " ";
        if (n->left) q.push(n->left);
        if (n->right) q.push(n->right);
    }
}

void destroy(Node* n) { if (!n) return; destroy(n->left); destroy(n->right); delete n; }  // postorder!

int main() {
    Node* root = new Node('A', new Node('B', new Node('D'),
                          new Node('E', new Node('G'), new Node('H'))),
                          new Node('C', nullptr, new Node('F')));
    inorder(root); cout << endl;      // D B G E H A C F
    levelOrder(root); cout << endl;   // A B C D E F G H
    cout << height(root) << endl;     // 3
    destroy(root);
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Sum, Maximum and Leaf Values
The notebook provides `TreeNode` and `build_tree(level_order_list)`. Write recursive functions `tree_sum(root)`, `tree_max(root)` (return `None` for an empty tree) and `leaf_values(root)` (left-to-right list of leaf data).

```text
build_tree([5, 3, 8, 1, 4, None, 9]) → sum 30, max 9, leaves [1, 4, 9]
```

In [ ]:
class TreeNode:
    def __init__(self, data, left=None, right=None):
        self.data = data
        self.left = left
        self.right = right

def build_tree(values):
    """Build a binary tree from a level-order list; None means 'no node'."""
    if not values or values[0] is None:
        return None
    from collections import deque
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def tree_sum(root):
    # TODO
    pass

def tree_max(root):
    # TODO: None for an empty tree
    pass

def leaf_values(root):
    # TODO
    pass

In [ ]:
# Test cell: run after your solution
t = build_tree([5, 3, 8, 1, 4, None, 9])
assert tree_sum(t) == 30 and tree_max(t) == 9 and leaf_values(t) == [1, 4, 9]
assert tree_sum(None) == 0 and tree_max(None) is None and leaf_values(None) == []
assert tree_max(build_tree([-5, -9, -1])) == -1
print("✅ Task 1 passed")

### Task 2 🟢 Mirror (Invert) a Binary Tree
Write `mirror(root)` that swaps the left and right child of **every** node and returns the root. Check your answer with level-order traversal.

```text
[4, 2, 7, 1, 3, 6, 9] → level order [4, 7, 2, 9, 6, 3, 1]
```

In [ ]:
def mirror(root):
    # TODO
    return root

In [ ]:
# Test cell: run after your solution
from collections import deque
def lo(r):
    out, q = [], deque([r] if r else [])
    while q:
        n = q.popleft(); out.append(n.data)
        q.extend(c for c in (n.left, n.right) if c)
    return out
assert lo(mirror(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [4, 7, 2, 9, 6, 3, 1]
assert mirror(None) is None
print("✅ Task 2 passed")

### Task 3 🟡 Same Tree and Symmetric Tree
Write `same_tree(a, b)` (same shape and same values) and `is_symmetric(root)` (the tree is a mirror image of itself around its centre). Hint: a tree is symmetric if its left subtree is the mirror of its right subtree; write a helper `is_mirror(x, y)`.

```text
[1, 2, 2, 3, 4, 4, 3] → symmetric
[1, 2, 2, None, 3, None, 3] → not symmetric
```

In [ ]:
def same_tree(a, b):
    # TODO
    pass

def is_mirror(x, y):
    # TODO
    pass

def is_symmetric(root):
    # TODO
    pass

In [ ]:
# Test cell: run after your solution
assert same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3])) is True
assert same_tree(build_tree([1, 2]), build_tree([1, None, 2])) is False
assert same_tree(None, None) is True
assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])) is True
assert is_symmetric(build_tree([1, 2, 2, None, 3, None, 3])) is False
assert is_symmetric(None) is True
print("✅ Task 3 passed")

### Task 4 🟡 Level-by-Level Report
Write `levels(root)` that returns a list of lists: the values at each level, from top to bottom. Then write `right_view(root)`: the last value of each level (what you see when looking at the tree from the right side).

```text
[3, 9, 20, None, None, 15, 7] → levels [[3], [9, 20], [15, 7]], right view [3, 20, 7]
```

In [ ]:
from collections import deque

def levels(root):
    # TODO: BFS; process len(q) nodes per level
    pass

def right_view(root):
    # TODO
    pass

In [ ]:
# Test cell: run after your solution
t = build_tree([3, 9, 20, None, None, 15, 7])
assert levels(t) == [[3], [9, 20], [15, 7]] and right_view(t) == [3, 20, 7]
assert levels(None) == [] and right_view(None) == []
assert right_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]
print("✅ Task 4 passed")

### Task 5 🟡 Iterative Inorder Traversal
Write `inorder_iter(root)` **without recursion**, using an explicit stack. Hint: go left as far as possible pushing nodes; pop a node, visit it, then move to its right child.

```text
[1, None, 2, 3] → [1, 3, 2]
```

In [ ]:
def inorder_iter(root):
    out, stack, cur = [], [], root
    # TODO
    return out

In [ ]:
# Test cell: run after your solution
import random
def rec(r):
    return rec(r.left) + [r.data] + rec(r.right) if r else []
assert inorder_iter(build_tree([1, None, 2, 3])) == [1, 3, 2]
assert inorder_iter(None) == []
for _ in range(10):
    vals = [random.choice([None] + list(range(9))) for _ in range(20)]; vals[0] = 1
    t = build_tree(vals)
    assert inorder_iter(t) == rec(t)
print("✅ Task 5 passed")

### Task 6 🔴 Diameter of a Binary Tree
The **diameter** is the number of edges on the longest path between **any two nodes** (the path may or may not pass through the root). A naive solution computes height at every node (O(n²)).

Write an **O(n)** solution with a helper that returns the height of a subtree while updating the best diameter seen so far.

```text
[1, 2, 3, 4, 5] → 3   (path 4 → 2 → 1 → 3)
```

In [ ]:
def diameter(root):
    best = 0
    def height(node):
        nonlocal best
        # TODO: compute left and right heights, update best, return height
        pass
    height(root)
    return best

In [ ]:
# Test cell: run after your solution
assert diameter(build_tree([1, 2, 3, 4, 5])) == 3
assert diameter(build_tree([1, 2])) == 1
assert diameter(None) == 0 and diameter(build_tree([7])) == 0
assert diameter(build_tree([1, 2, None, 3, 4, 5, None, None, 6, 7, None, None, 8])) == 6
print("✅ Task 6 passed")

### Task 7 🔴 Rebuild a Tree from Traversals
Given the **preorder** and **inorder** traversals of a binary tree with unique values, rebuild the tree and return its root.

Key idea: the first preorder value is the root; its position in inorder splits the left and right subtrees. Use a dictionary of inorder positions for O(n).

```text
preorder [3, 9, 20, 15, 7], inorder [9, 3, 15, 20, 7] → tree [3, 9, 20, None, None, 15, 7]
```

In [ ]:
def rebuild(preorder, inorder):
    pos = {v: i for i, v in enumerate(inorder)}
    it = iter(preorder)

    def build(lo, hi):
        # TODO: take the next preorder value as root; split inorder at its position
        pass

    return build(0, len(inorder) - 1)

In [ ]:
# Test cell: run after your solution
def pre(r): return [r.data] + pre(r.left) + pre(r.right) if r else []
def ino(r): return ino(r.left) + [r.data] + ino(r.right) if r else []
t = rebuild([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
assert pre(t) == [3, 9, 20, 15, 7] and ino(t) == [9, 3, 15, 20, 7] and t.right.data == 20
assert rebuild([], []) is None
import random
for _ in range(10):
    vals = random.sample(range(100), 15); orig = build_tree(vals)
    r = rebuild(pre(orig), ino(orig)); assert pre(r) == pre(orig) and ino(r) == ino(orig)
print("✅ Task 7 passed")

## 🔬 Bonus: A Real Decision Tree (Machine Learning)

In [ ]:
# Bonus: run after completing the tasks
try:
    from sklearn.datasets import load_iris
    from sklearn.tree import DecisionTreeClassifier, export_text
    iris = load_iris()
    model = DecisionTreeClassifier(max_depth=2, random_state=0).fit(iris.data, iris.target)
    print(export_text(model, feature_names=list(iris.feature_names)))
    print("Each internal node asks a question; each leaf gives a prediction: a binary tree!")
except ImportError:
    print("scikit-learn is not installed here; run this cell in Google Colab.")

---
## 🎉 Lab 13 Complete!

Next: **Lab 14: Binary Search Trees**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab13/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)